#### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. 
- A tool is typically defined with three critical pieces of metadata (Name, Description, Args Schema / Function Signature - often a JSON format).
- No Real Time Data and No Action Taking capability, Computational Limitations - they often fail at logical data and precise arithmetic



In [4]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")   
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the way humans use language, but many species are excellent vocal mimics. Their ability to copy human speech (and other sounds) is the result of a mix of anatomy, brain wiring, social needs, and evolutionary history. Here’s a breakdown of why parrots can and often do imitate words:\n\n---\n\n## 1. Anatomy: A “built‑in” sound‑production system\n\n| Feature | What it does | Why it matters for mimicry |\n|---------|--------------|---------------------------|\n| **Syrinx** (the bird’s voice box) | Located at the base of the trachea, the syrinx contains two sets of vibrating membranes that can be controlled independently. | Allows parrots to produce a huge variety of tones, pitches, and rapid modulations—far more than most other birds. |\n| **Highly mobile tongue & beak** | Parrots have a strong, dexterous tongue and a hooked beak that can shape airflow. | Fine‑tunes the acoustic quality of each sound, helping them match the subtle nuances of human

In [ ]:
from langchain.tools import tool

@tool    #decorator syntax - Python fucntion to Structured tool object that an LLM powered agent can understand, reason about and execute
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

# tool need to be binded with the model
model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We need to retrieve weather via function get_weather. Use function call.', 'tool_calls': [{'id': 'fc_b4d72dcc-4eef-4c16-bd40-4adbdb155e19', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 127, 'total_tokens': 181, 'completion_time': 0.114904632, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.005076306, 'prompt_tokens_details': None, 'queue_time': 0.317375804, 'total_time': 0.119980938}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fe269835c7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1e322-9e7c-7613-82bb-8011b738a321-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_b4d72dcc-4eef-4c16-bd40-4adbdb155e19', 'type

#### Tool Execution Loops

In [9]:
# model genarets the tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# execute tool and collect the result
for too_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Boston is sunny. 🌞 Let me know if you’d like a forecast for the next few days or any other details!


In [11]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_541054c1-3eff-416a-9c8c-9727ff9ae5c7', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 126, 'total_tokens': 165, 'completion_time': 0.08200683, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.005177836, 'prompt_tokens_details': None, 'queue_time': 0.368805713, 'total_time': 0.087184666}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1e32f-70f0-7728-9374-b1e9c8553028-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_541054c1-3eff-416a-9c8c-9727ff9ae5c7',